# 02 — Experiment Tracking con MLflow

En este notebook aprenderemos a usar **MLflow** para registrar, comparar y reproducir experimentos de machine learning.

## ¿Qué es MLflow?

MLflow es una plataforma open-source para gestionar el ciclo de vida completo de modelos de ML. Tiene **4 componentes principales**:

| Componente | ¿Qué hace? |
|---|---|
| **Tracking** | Registra parámetros, métricas y artefactos de cada experimento |
| **Projects** | Empaqueta código para reproducibilidad |
| **Models** | Define un formato estándar para servir modelos (flavors) |
| **Registry** | Gestiona el ciclo de vida de modelos: versiones, stages (Staging/Production) |

In [ ]:
!pip install mlflow lightgbm pandas numpy scikit-learn plotly -q

In [ ]:
import mlflow
import mlflow.lightgbm
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from mlflow.tracking import MlflowClient
from mlflow.models.signature import infer_signature
import warnings
warnings.filterwarnings('ignore')
import os

# En el contenedor de Jupyter (make up) los runs van al servidor de MLflow
# (UI: http://localhost:5050). Fuera de Docker se guardan en ./mlruns.
mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "mlruns"))
print(f"MLflow version: {mlflow.__version__}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

## 1. Arquitectura de MLflow

```
┌─────────────────────────────────────────────────────────┐
│                     MLflow Platform                      │
│                                                          │
│  ┌─────────────┐  ┌───────────┐  ┌───────┐  ┌───────┐ │
│  │  Tracking   │  │ Projects  │  │Models │  │Registry│ │
│  │             │  │           │  │       │  │       │  │
│  │ Experiments │  │Reproducib.│  │Flavors│  │Stages  │  │
│  │ Runs        │  │Packaging  │  │Deploy │  │Versions│  │
│  │ Params      │  │           │  │Serve  │  │       │  │
│  │ Metrics     │  │           │  │       │  │       │  │
│  │ Artifacts   │  │           │  │       │  │       │  │
│  └─────────────┘  └───────────┘  └───────┘  └───────┘  │
│                                                          │
│  Backend Store              Artifact Store               │
│  (SQLite/PostgreSQL)        (Local/S3/MinIO)             │
└─────────────────────────────────────────────────────────┘
```

- **Backend Store**: guarda metadatos (parámetros, métricas, tags). En local usa archivos; en producción, una base de datos.
- **Artifact Store**: guarda archivos grandes (modelos, plots, datasets). En local usa el filesystem; en producción, S3 o MinIO.

In [ ]:
rng = np.random.default_rng(42)
n = 2000
hours = rng.integers(0, 24, n)
temps = rng.normal(26, 5, n)
demand = (
    1000 + 80 * np.sin(2 * np.pi * hours / 24)
    + 3 * temps + rng.normal(0, 50, n)
)

X = np.column_stack([hours, temps, (hours >= 18) & (hours <= 21)])
y = demand

X_train, X_test = X[:1600], X[1600:]
y_train, y_test = y[:1600], y[1600:]
feature_names = ["hour", "temperature", "is_peak"]
print(f"Train: {len(X_train)} | Test: {len(X_test)}")

## 2. Tu primer run MLflow

Un **run** es la unidad básica de MLflow: cada vez que entrenas un modelo (o pruebas una configuración), creas un run.

Dentro de un run puedes loguear:
- `mlflow.log_param()` — hiperparámetros y configuración
- `mlflow.log_metric()` — resultados numéricos (RMSE, accuracy, etc.)
- `mlflow.set_tag()` — metadatos libres (autor, dataset, etc.)
- `mlflow.log_artifact()` — archivos (modelos, plots, CSV)

In [ ]:
with mlflow.start_run(run_name="primer_run_manual") as run:
    # Loguear parámetros (configuración del experimento)
    mlflow.log_param("model_type", "manual_test")
    mlflow.log_param("n_features", 3)
    mlflow.log_param("train_size", len(X_train))

    # Loguear métricas (resultados)
    mlflow.log_metric("rmse_train", 55.3)
    mlflow.log_metric("rmse_val", 62.1)
    mlflow.log_metric("r2", 0.87)

    # Loguear tags (metadatos)
    mlflow.set_tag("author", "tu-nombre")
    mlflow.set_tag("dataset", "synthetic_demand")

    run_id = run.info.run_id

print(f"Run completado!")
print(f"Run ID: {run_id}")
print("Ver el run en la UI de MLflow: http://localhost:5050")

## 3. ¿Qué ves en la UI?

Para abrir la UI de MLflow:
- **Con el stack de Docker (`make up`)**: abrir http://localhost:5050
- **Sin Docker** (runs en `./mlruns`): ejecutar `mlflow ui` en la carpeta del notebook y abrir http://localhost:5000

Lo que verás:

```
┌─────────────────────────────────────────────────────────────┐
│  MLflow                                    Experiments  Models│
├─────────────────────────────────────────────────────────────┤
│  Experiment: Default                                          │
│  ┌──────────────────┬──────────┬──────┬────────┬──────────┐ │
│  │ Run Name         │ Created  │ rmse │ model  │ author   │ │
│  ├──────────────────┼──────────┼──────┼────────┼──────────┤ │
│  │ primer_run_manual│ just now │ 62.1 │manual  │ tu-nombre│ │
│  └──────────────────┴──────────┴──────┴────────┴──────────┘ │
│                                                               │
│  Click en un run → verás: Params | Metrics | Tags | Artifacts│
└─────────────────────────────────────────────────────────────┘
```

## 4. Entrenamiento real con LightGBM

Ahora entrenamos un modelo real y dejamos que MLflow registre automáticamente los parámetros del modelo.

Con `mlflow.lightgbm.log_model()` guardamos el modelo completo como artefacto, incluyendo su **firma** (schema de inputs/outputs) para validación automática.

In [ ]:
import pandas as pd

X_train_df = pd.DataFrame(X_train, columns=feature_names)
X_test_df  = pd.DataFrame(X_test, columns=feature_names)

params = {
    "n_estimators": 200,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "random_state": 42,
    "verbose": -1,
}

mlflow.set_experiment("demand_forecasting_demo")

with mlflow.start_run(run_name="lgbm_v1") as run:
    mlflow.log_params(params)

    model = lgb.LGBMRegressor(**params)
    model.fit(X_train_df, y_train, eval_set=[(X_test_df, y_test)],
              callbacks=[lgb.early_stopping(20, verbose=False)])

    y_pred = model.predict(X_test_df)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("n_estimators_used", model.best_iteration_)

    # Loguear modelo con firma
    signature = infer_signature(X_train_df, model.predict(X_train_df))
    mlflow.lightgbm.log_model(model, "model", signature=signature)

    run_id_v1 = run.info.run_id

print(f"RMSE: {rmse:.2f}")
print(f"Run ID: {run_id_v1}")

## 5. Comparar 3 runs con distintos hiperparámetros

Una de las ventajas de MLflow es poder lanzar múltiples runs y compararlos visualmente en la UI. Aquí probamos 3 configuraciones distintas de LightGBM.

In [ ]:
configs = [
    {"n_estimators": 100, "learning_rate": 0.1,  "num_leaves": 15},
    {"n_estimators": 300, "learning_rate": 0.05, "num_leaves": 63},
    {"n_estimators": 500, "learning_rate": 0.01, "num_leaves": 127},
]

run_ids = []
for i, cfg in enumerate(configs):
    cfg["random_state"] = 42
    cfg["verbose"] = -1
    with mlflow.start_run(run_name=f"config_{i+1}") as run:
        mlflow.log_params(cfg)
        model = lgb.LGBMRegressor(**cfg)
        model.fit(X_train_df, y_train, eval_set=[(X_test_df, y_test)],
                  callbacks=[lgb.early_stopping(20, verbose=False)])
        y_pred = model.predict(X_test_df)
        rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
        mlflow.log_metric("rmse", rmse)
        # Se guarda el modelo en cada run para poder cargar el mejor más adelante
        mlflow.lightgbm.log_model(model, "model", signature=infer_signature(X_train_df, y_pred))
        run_ids.append(run.info.run_id)
        print(f"Config {i+1}: RMSE={rmse:.2f} | n_estimators={cfg['n_estimators']} | lr={cfg['learning_rate']}")

## 6. Comparar runs programáticamente con MlflowClient

`MlflowClient` permite interactuar con el servidor de MLflow desde Python: buscar runs, leer métricas, gestionar el Model Registry, etc.

In [ ]:
client = MlflowClient()

# Buscar todos los runs del experimento
experiment = mlflow.get_experiment_by_name("demand_forecasting_demo")
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.rmse ASC"],
)

print("Todos los runs (ordenados por RMSE):")
print(runs[["run_id", "params.n_estimators", "params.learning_rate", "metrics.rmse"]].to_string(index=False))

best_run = runs.iloc[0]
print(f"\nMejor run: {best_run['run_id'][:8]}... con RMSE={best_run['metrics.rmse']:.2f}")

## 7. Cargar un modelo desde un run

MLflow guarda los modelos como artefactos versionados. Se pueden cargar usando el `run_id` o una URI del Model Registry.

El formato `runs:/<run_id>/model` es la forma estándar de referenciar un modelo guardado.

In [ ]:
best_run_id = runs.iloc[0]["run_id"]
model_uri = f"runs:/{best_run_id}/model"

print(f"Cargando modelo desde: {model_uri}")
loaded_model = mlflow.pyfunc.load_model(model_uri)

sample = X_test_df.head(5)
predictions = loaded_model.predict(sample)
print("\nSample predictions:")
for i, (real, pred) in enumerate(zip(y_test[:5], predictions)):
    print(f"  Real: {real:.1f} kWh | Predicción: {pred:.1f} kWh | Error: {abs(real-pred):.1f}")

## 8. Conectarse a un servidor MLflow remoto

En producción, MLflow corre como un servidor centralizado al que todos los miembros del equipo se conectan. El servidor almacena:
- **Metadatos** en una base de datos (PostgreSQL)
- **Artefactos** en un object store (MinIO/S3)

Solo cambiando el `tracking_uri` tu código existente funciona igual, pero ahora los runs son visibles para todo el equipo.

In [ ]:
# Dentro del contenedor de Jupyter (docker compose) el servidor es:
# mlflow.set_tracking_uri("http://mlflow-server:5000")

# Desde tu máquina, con el stack levantado (make up):
# mlflow.set_tracking_uri("http://localhost:5050")

# Los artefactos los sirve el propio servidor de MLflow (--serve-artifacts),
# así que no hace falta configurar credenciales de S3.

print("Tracking URI actual:", mlflow.get_tracking_uri())
print("Desde tu máquina, el servidor del docker-compose es:")
print('  mlflow.set_tracking_uri("http://localhost:5050")')

## Resumen: ¿Qué aprendimos?

- **MLflow Tracking** registra automáticamente parámetros, métricas, tags y artefactos de cada run
- **Experiments** agrupan runs relacionados para facilitar la comparación
- **`mlflow.start_run()`** es el contexto manager que delimita un experimento
- **`mlflow.log_model()`** guarda el modelo como artefacto con su firma (schema de inputs/outputs)
- **`MlflowClient`** permite consultar runs y métricas programáticamente para toma de decisiones
- **`mlflow.pyfunc.load_model()`** carga cualquier modelo de forma agnóstica al framework
- Cambiar **`tracking_uri`** es suficiente para pasar de desarrollo local a un servidor compartido en producción
- La **UI web** (`mlflow ui`) permite comparar runs visualmente sin escribir código